# Regime-Aware Commodity Allocator
## Build a contextual bandit with real market data

**Goal:** Build a practical commodity allocation system that adapts to market regimes using LinUCB with real features.

**Key Insight:** Context features (volatility, term structure, trend) transform a static allocation into a regime-aware system that automatically adjusts to market conditions.

**Time:** 15 minutes

In [ ]:
callout("** Context features (volatility, term structure, trend) transform a static allocation into a regime-aware system that automatically adjusts to market", kind="insight")

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timedelta

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries loaded successfully!")

In [ ]:
learning_objectives(["**Context transforms static allocation** — By observing volatility, trend, and seasonality, the bandit adapts dynamically", "**Feature engineering is critical** — Good features (vol regime, trend) enable learning; bad features add noise", "**Interpretability matters** — Learned weights show which features drive each commodity's performance", "**Real-world performance** — Contextual bandit can outperform equal-weight by adapting to regime shifts", "**Exploration-exploitation balance** — LinUCB automatically explores during uncertain regimes and exploits during familiar ones"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## Cell 2: Load Real Commodity Data

Get 3 commodity sectors: Energy (CL=F), Metals (GC=F), Agriculture (ZC=F)

In [ ]:
# Download commodity futures data
tickers = {
    'Energy': 'CL=F',        # WTI Crude Oil
    'Metals': 'GC=F',        # Gold
    'Agriculture': 'ZC=F'    # Corn
}

# Get 3 years of data
end_date = datetime.now()
start_date = end_date - timedelta(days=3*365)

prices = pd.DataFrame()
for name, ticker in tickers.items():
    try:
        data = yf.download(ticker, start=start_date, end=end_date, progress=False)
        prices[name] = data['Adj Close']
    except:
        print(f"Warning: Could not download {ticker}, using synthetic data")
        dates = pd.date_range(start=start_date, end=end_date, freq='D')
        prices[name] = pd.Series(100 * (1 + np.random.randn(len(dates)).cumsum() * 0.01), index=dates)

prices = prices.dropna()
returns = prices.pct_change().dropna()

print(f"Data shape: {prices.shape}")
print(f"Date range: {prices.index[0]} to {prices.index[-1]}")
print(f"\nPrice statistics:")
print(prices.describe())

## Cell 3: Compute Context Features

Extract market regime features: volatility, trend, seasonality

In [ ]:
def extract_context_features(prices, returns):
    """Extract regime features from commodity data."""
    features = pd.DataFrame(index=prices.index)
    
    # 1. Volatility regime (20-day realized vol, z-scored)
    vol = returns.std(axis=1).rolling(20).mean()
    features['vol_zscore'] = (vol - vol.mean()) / vol.std()
    
    # 2. Trend strength (50-day vs 200-day MA)
    price_avg = prices.mean(axis=1)
    ma_50 = price_avg.rolling(50).mean()
    ma_200 = price_avg.rolling(200).mean()
    trend = (ma_50 - ma_200) / ma_200
    features['trend_zscore'] = (trend - trend.mean()) / trend.std()
    
    # 3. Seasonality (cyclical encoding)
    month = features.index.month
    features['season_sin'] = np.sin(2 * np.pi * month / 12)
    features['season_cos'] = np.cos(2 * np.pi * month / 12)
    
    # 4. Recent momentum (20-day return)
    mom = price_avg.pct_change(20)
    features['momentum_zscore'] = (mom - mom.mean()) / mom.std()
    
    return features.fillna(0)

features = extract_context_features(prices, returns)

print("Context features:")
print(features.head())
print(f"\nFeature correlations:")
print(features.corr())

## Cell 4: Implement Contextual Bandit Allocator

Use LinUCB to choose which commodity sector to overweight.

In [ ]:
class LinUCB:
    """Linear Upper Confidence Bound."""
    
    def __init__(self, n_arms, context_dim, alpha=1.0, lambda_=1.0):
        self.n_arms = n_arms
        self.d = context_dim
        self.alpha = alpha
        self.A = [lambda_ * np.eye(context_dim) for _ in range(n_arms)]
        self.b = [np.zeros(context_dim) for _ in range(n_arms)]
    
    def choose_arm(self, context):
        ucb_scores = []
        for a in range(self.n_arms):
            theta = np.linalg.solve(self.A[a], self.b[a])
            pred = context @ theta
            std = np.sqrt(context @ np.linalg.solve(self.A[a], context))
            ucb_scores.append(pred + self.alpha * std)
        return np.argmax(ucb_scores)
    
    def update(self, arm, context, reward):
        self.A[arm] += np.outer(context, context)
        self.b[arm] += reward * context
    
    def get_weights(self, arm):
        return np.linalg.solve(self.A[arm], self.b[arm])

print("LinUCB allocator ready!")

## Cell 5: Backtest Regime-Aware vs Fixed Allocation

Compare contextual bandit to equal-weight baseline.

In [ ]:
# Align data
valid_idx = features.index.intersection(returns.index)
features_aligned = features.loc[valid_idx]
returns_aligned = returns.loc[valid_idx]

# Skip initial period for feature computation
start_idx = 250
features_bt = features_aligned.iloc[start_idx:]
returns_bt = returns_aligned.iloc[start_idx:]

# Initialize bandit
n_arms = 3
context_dim = features_bt.shape[1]
bandit = LinUCB(n_arms=n_arms, context_dim=context_dim, alpha=1.0)

# Run backtest
portfolio_returns_bandit = []
portfolio_returns_equal = []
arm_choices = []

for i in range(len(features_bt)):
    # Get context
    context = features_bt.iloc[i].values
    
    # Bandit chooses arm
    arm = bandit.choose_arm(context)
    arm_choices.append(arm)
    
    # Observe returns (next period)
    if i < len(returns_bt) - 1:
        next_returns = returns_bt.iloc[i + 1].values
        
        # Bandit portfolio: 100% chosen arm
        bandit_return = next_returns[arm]
        portfolio_returns_bandit.append(bandit_return)
        
        # Equal-weight portfolio
        equal_return = np.mean(next_returns)
        portfolio_returns_equal.append(equal_return)
        
        # Update bandit with realized return
        bandit.update(arm, context, bandit_return)

# Convert to series
portfolio_returns_bandit = pd.Series(portfolio_returns_bandit, index=features_bt.index[:-1])
portfolio_returns_equal = pd.Series(portfolio_returns_equal, index=features_bt.index[:-1])

print(f"Backtest period: {len(portfolio_returns_bandit)} days")
print(f"\nPerformance Summary:")
print(f"Contextual Bandit Total Return: {(1 + portfolio_returns_bandit).prod() - 1:.2%}")
print(f"Equal Weight Total Return: {(1 + portfolio_returns_equal).prod() - 1:.2%}")
print(f"Outperformance: {((1 + portfolio_returns_bandit).prod() - (1 + portfolio_returns_equal).prod()):.2%}")

## Cell 6: Visualize Regime Adaptation

See how the bandit switches allocations based on market conditions.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 1. Cumulative returns
cum_bandit = (1 + portfolio_returns_bandit).cumprod()
cum_equal = (1 + portfolio_returns_equal).cumprod()

axes[0].plot(cum_bandit, label='Contextual Bandit', linewidth=2)
axes[0].plot(cum_equal, label='Equal Weight', linewidth=2, alpha=0.7)
axes[0].set_ylabel('Cumulative Return')
axes[0].set_title('Contextual Bandit Adapts to Market Regimes')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Arm selection over time
arm_series = pd.Series(arm_choices[:-1], index=features_bt.index[:-1])
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
arm_names = list(tickers.keys())

for arm in range(3):
    mask = arm_series == arm
    axes[1].scatter(arm_series.index[mask], [arm]*mask.sum(), 
                   c=colors[arm], label=arm_names[arm], alpha=0.6, s=10)

axes[1].set_ylabel('Selected Arm')
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(arm_names)
axes[1].set_title('Allocation Switches Based on Market Regime')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# 3. Volatility regime (context feature)
vol_context = features_bt['vol_zscore'].iloc[:-1]
axes[2].plot(vol_context, linewidth=1.5, color='purple', alpha=0.7)
axes[2].axhline(y=0, color='k', linestyle='--', linewidth=1)
axes[2].fill_between(vol_context.index, 0, vol_context, 
                      where=(vol_context > 0), alpha=0.3, color='red', label='High Vol')
axes[2].fill_between(vol_context.index, 0, vol_context,
                      where=(vol_context <= 0), alpha=0.3, color='green', label='Low Vol')
axes[2].set_ylabel('Volatility Z-Score')
axes[2].set_xlabel('Date')
axes[2].set_title('Market Volatility Regime')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nNotice how arm selection changes with market regimes!")

## Cell 7: Analyze Learned Feature Weights

Understand what the bandit learned about each commodity sector.

In [ ]:
# Extract learned weights
learned_weights = pd.DataFrame(
    [bandit.get_weights(a) for a in range(3)],
    index=arm_names,
    columns=features_bt.columns
)

print("Learned Feature Weights:")
print(learned_weights)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
learned_weights.T.plot(kind='bar', ax=ax, width=0.8)
ax.set_ylabel('Weight')
ax.set_xlabel('Feature')
ax.set_title('LinUCB Learned Weights: Which Features Drive Each Commodity?')
ax.legend(title='Commodity')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Positive weight: Commodity performs better when feature is high")
print("- Negative weight: Commodity performs better when feature is low")
print("\nExample insights:")
for arm in range(3):
    top_feature = learned_weights.iloc[arm].abs().idxmax()
    weight = learned_weights.iloc[arm][top_feature]
    direction = "high" if weight > 0 else "low"
    print(f"- {arm_names[arm]}: Favors {direction} {top_feature}")

## Cell 8: Modify This — Add New Features

Experiment with adding more context features to improve performance.

In [ ]:
def add_macro_features(features, returns):
    """Add macro regime features (simplified proxies)."""
    # Dollar proxy: inverse correlation with commodities
    avg_return = returns.mean(axis=1)
    dollar_proxy = -avg_return.rolling(20).mean()
    features['dollar_proxy'] = (dollar_proxy - dollar_proxy.mean()) / dollar_proxy.std()
    
    # Risk sentiment: rolling correlation between commodities
    risk_on = returns.iloc[:, 0].rolling(20).corr(returns.iloc[:, 1])
    features['risk_on'] = (risk_on - risk_on.mean()) / risk_on.std()
    
    return features.fillna(0)

# Test with enhanced features
features_enhanced = extract_context_features(prices, returns)
features_enhanced = add_macro_features(features_enhanced, returns)

print("Enhanced features:")
print(features_enhanced.columns.tolist())
print(f"\nFeature dimension increased: {context_dim} → {features_enhanced.shape[1]}")
print("\nModify This:")
print("- Add VIX data as a real volatility feature")
print("- Include inventory surprise data (EIA for energy)")
print("- Add weather indicators for agriculture")
print("- Try different lookback windows (10-day vs 50-day vol)")
print("- Experiment with feature selection (drop low-importance features)")

## Summary and Key Takeaways

**What we built:**
A production-grade regime-aware commodity allocator using LinUCB with real market data and features.

**Key insights:**

1. **Context transforms static allocation** — By observing volatility, trend, and seasonality, the bandit adapts dynamically

2. **Feature engineering is critical** — Good features (vol regime, trend) enable learning; bad features add noise

3. **Interpretability matters** — Learned weights show which features drive each commodity's performance

4. **Real-world performance** — Contextual bandit can outperform equal-weight by adapting to regime shifts

5. **Exploration-exploitation balance** — LinUCB automatically explores during uncertain regimes and exploits during familiar ones

**Practical considerations:**

- **Transaction costs:** High switching frequency can erode gains; consider smoothing or minimum hold periods
- **Feature lag:** Ensure all features are observable before decision time (no lookahead bias)
- **Non-stationarity:** Market regimes change; consider decay or adaptive methods (Module 6)
- **Risk management:** Add constraints (max leverage, sector limits) around the bandit

**Next steps:**
- Module 4: Apply contextual bandits to content and growth optimization
- Module 5: Build production commodity systems with guardrails
- Module 6: Handle non-stationary environments with change detection

**Extension ideas:**
- Portfolio optimization: Use bandit to tilt weights around a baseline portfolio
- Signal routing: Route decisions to different forecasting models based on regime
- Multi-asset: Extend to equities, FX, crypto with asset-class specific features
- Thompson Sampling: Try Bayesian alternative to LinUCB for better uncertainty calibration

---

*"The market doesn't care what worked last month. Context-aware allocation adapts to what works now."*

In [ ]:
key_takeaways(["Transaction costs:", "Feature lag:", "Non-stationarity:", "Risk management:"])